In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
sns.set_style('whitegrid')
sns.set_palette("Set2")

%matplotlib inline

# Leer los datos

In [2]:
totales = pd.read_csv("../../../data/respuestas_fede.csv")
print(totales.shape)

#globales
marmol = totales.loc[totales["escuela"] == "Colegio Modelo Mármol"]
mantovani = totales.loc[totales["escuela"] == "Escuela Nueva Juan Mantovani"]

file_ext = '.png'
dpi_value = 200

(369, 22)


## Qué es la nube vs Funcionalidad de la nube

In [3]:
# funcion aux para analizar misc
def analizarMiscFuncionalidad(val, agrupando = False):
    lista = val.split(",")
    misc = 0
    not_misc = 0
    if(lista == ["0"]): # Si no seleccionaron ninguna rta, es porque contestaron no se en la pregunta anterior
        return "Sin\nRta." 
    
    # respuestas sin misconception, # not_misc
    if "Lanubemepermiteguardarlasfotosyvideosdelcelular" in lista:
        not_misc = not_misc + 1
    if "InstagramyTikTokusanlanubeparacompartirsusvideos" in lista:
        not_misc = not_misc + 1
    if "Sepuedeusarlanubeparajugarjuegossininstalarlos" in lista:
        not_misc = not_misc + 1
    if "GoogleMapsdescargasusmapasdelanube" in lista:
        not_misc = not_misc + 1

    # respuestas con misconception, # misc
    if "PodemosutilizarlanubesinconexiónaInternet" in lista:
        misc = misc + 1
    if "Sinlanubeseríaimposibleescucharmúsicaenelcelular" in lista:
        misc = misc + 1
    if "Sinlanubenoseríaposiblehacerllamadasporcelular" in lista:
        misc = misc + 1
    if "Sinlanubeseríaimposiblesacarfotosconelcelular" in lista:
        misc = misc + 1
    
    if(not agrupando):
        if(not_misc == misc):
            return "Iguales\nMisc. y\nNo Misc."    
        if(misc == 0):
            return "Ninguna\nMisc."        
        if(not_misc == 0):
            return "Todas\nMisc."      
        if(not_misc > misc):        
            return "Minoría\nMisc."    
        if(misc > not_misc):
            return "Mayoría\nMisc."
    else:
        if(not_misc == misc):
            return "Iguales\nMisc. y\nNo Misc."      
        if(not_misc > misc):        
            return "Minoría\nMisc."    
        if(misc > not_misc):
            return "Mayoría\nMisc."

def analizarMiscNube(val):
    if val == "Una nube":
        return "Misc. fuerte"
    if val == "Una parte del celular":
        return "Misc. fuerte"
    if val == "Una computadora gigante":
        return "Misc. fuerte"
    if val == "Una red de antenas y satélites":
        return "Misc. débil"
    if val == "Muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)":
        return "Sin Misc."
    

In [4]:
# Boilerplate para generar grafico de nube vs func de nube
def nube_vs_funcionalidad(df, df_name, agrupando = False):
    que_es_nube = df["que_es_nube"]
    que_es_nube_sin_nose = que_es_nube[que_es_nube != "No sé"]

    misconceptions_funcionalidad_nube = df["afirmaciones_nube"].str.replace(" ", "").fillna("0").apply(lambda x: analizarMiscFuncionalidad(x,agrupando))
    misconceptions_funcionalidad_nube_sin_sinrta = misconceptions_funcionalidad_nube[misconceptions_funcionalidad_nube != "Sin\nRta."]

    if(agrupando):
        filas = [
                 "Misc. fuerte",
                 "Misc. débil",
                 "Sin Misc.",
                ]
        columnas = [
                    "Minoría\nMisc.",
                    "Iguales\nMisc. y\nNo Misc.",
                    "Mayoría\nMisc.",
                   ]
        que_es_nube_sin_nose = que_es_nube_sin_nose.apply(analizarMiscNube)
        
    else:
        filas = [
                 "Una nube",
                 "Una parte del celular",
                 "Una compu gigante", 
                 "Una red de antenas y satélites",
                 "Muchísimas compus",
                ]
        columnas = [
                    "Ninguna\nMisc.",
                    "Minoría\nMisc.",
                    "Iguales\nMisc. y\nNo Misc.",
                    "Mayoría\nMisc.",
                    "Todas\nMisc.",
                   ]

    cross_tab_result = pd.crosstab(que_es_nube_sin_nose, misconceptions_funcionalidad_nube_sin_sinrta)

    cross_tab_result = cross_tab_result.rename(index={"Muchísimas computadoras (tantas que podrían llenar una cancha de fútbol)": "Muchísimas compus",
                                "Una computadora gigante":"Una compu gigante"})

    # # filas   = arriba hacia abajo, mayores misconceptions a correcta
    # # columnas= izq a derecha,      correcta a mayores misconceptions
    cross_tab_result = cross_tab_result.reindex(filas, columns=columnas)

    def agregar_totales(fila):
       if(agrupando):
            return fila[0]+fila[1]+fila[2]
       else:
            return fila[0]+fila[1]+fila[2]+fila[3]+fila[4]
    
    def dividir(fila): 
       if(agrupando):
           return fila.div(fila[3])
       else:
           return fila.div(fila[5])

    cross_tab_result['totales'] = cross_tab_result.apply(agregar_totales, axis=1)
    cross_tab_result = cross_tab_result.apply(dividir, axis=1)
    cross_tab_result = cross_tab_result.drop(['totales'], axis=1)

    sns.heatmap(cross_tab_result, cmap='YlGnBu', annot=True, fmt='.2f', linewidths=0.5)
    plt.title('Relación de Respuestas\nQue es la nube vs Funcionalidad de la nube\nDatos {} - Agrupando {}'.format(df_name,agrupando))
    plt.xlabel('Misconceptions Funcionalidad Nube')
    plt.ylabel('Que es la nube')
    # plt.show()
    plt.savefig('nube_vs_func_{}_agrupando_{}'.format(df_name,agrupando)+file_ext, bbox_inches='tight', dpi=dpi_value)
    plt.clf()

In [5]:
# llamo al generador de graficos con los tres dfs
nube_vs_funcionalidad(totales, "totales")
nube_vs_funcionalidad(marmol, "marmol")
nube_vs_funcionalidad(mantovani, "mantovani")

# Agrupando misc
nube_vs_funcionalidad(totales, "totales", True)
nube_vs_funcionalidad(marmol, "marmol", True)
nube_vs_funcionalidad(mantovani, "mantovani", True)

<Figure size 640x480 with 0 Axes>